# NB-15 — SenForFlood under this paper's protocol (WP13: second dataset, event-disjoint)

**Why.** Reviewers 3 and 5 asked for validation on an independent dataset, and Reviewer 1 asked about the partition. Our existing SenForFlood numbers come from NB-09, which used a **different protocol** (cross-entropy + symmetric KL, lr 6e-5, 90 epochs, 256 crops, Hugging Face SegFormer). They must not be put in this paper. This notebook re-runs the SegFormer-B0 arm with the **Study 1 code and protocol** (lgmf v8: Focal-Tversky, AdamW 3e-4 / wd 1e-4, 45 epochs, batch 8, EMA 0.99, grad-clip 1, consistency λ 0.5, full 512 chips, flips only, seeds 42 / 1 / 2, selection on mean val IoU over the 3 regimes). Only the data loader is new.

**Data.** NB-08's output (`proc_senfor`): 1,339 chips from 28 CEMS events, split **by event**: train 801 chips (20 events) / val 269 (3) / test 269 (5). Channels VV, VH, B2, B3, B4, B8, B11, B12 (same 8 channels and order as Sen1Floods11). Labels −1 / 0 / 1, with permanent and flood water merged. Sentinel-2 here is L1C with clouds left in; **SAR is the strong sensor on this dataset and optical is the weak one** (Cohen's d: VV −1.82, VH −1.72, best optical B11 −0.70), the mirror of Sen1Floods11.

| Group | Runs (× 3 seeds) | Purpose |
|---|---|---|
| **A_recipe** (6 runs) | `sf_seg_off` (no dropout, no consistency = `seg_plain`), `sf_seg_on` (curriculum + consistency = `seg_full_ref`) | does the recipe transfer to a second dataset |
| *B_factorial* (6 runs, optional, off by default) | `sf_seg_drop_only` (curriculum, consistency 0), `sf_seg_cons_only` (no dropout, SAR-only consistency pass on every example = `seg_consonly`) | does "dropout and consistency are substitutes" (NB-12) hold on a second dataset |

**Rules fixed before looking at the results**
1. **No tuning on SenForFlood.** The recipe is exactly the Sen1Floods11 one. In particular the curriculum still ends at (0.40, 0.45, 0.15), which gives optical-only the smallest share even though optical is the weak sensor here. If that costs optical-only accuracy, the paper reports it as a limitation. Nothing is changed to rescue it.
2. "Helps" or "hurts" only if the difference (on − off, same seed) on **test** exceeds 0.03 in mean **and** has the same sign in all 3 seeds. Otherwise: "no detectable effect".
3. **Primary regime = optical-only** (the dependency is mirrored, and NB-09 already showed SAR-only needs no recipe here). This choice is motivated by NB-09 and is stated as such. SAR-only and both are reported with the same rule. A recipe effect is only claimed as "transfers" if the primary regime is classified "helps" and neither of the other two is classified "hurts".
4. Report the worst seed as well as the mean (NB-09 showed single-seed collapses, e.g. SegFormer optical-only 0.498 / 0.003 / 0.444).
5. Show the trivial floor (always-water IoU: the water fraction) next to every table, and the predicted-to-true water ratio, so a trivial solution cannot pass as a result.
6. Per-event numbers are reported for the 5 test events. 5 events is a small sample, so any statement is about those events.

**Tuning budget: none.** One configuration per run id.

**Attach** (Add Input): `lgmf_v8` · the **NB-08** notebook output (`proc_senfor`) · optional, for a side-by-side with Sen1Floods11: the NB-12 notebook output · on later commits, the previous commit of this notebook.
**Settings:** GPU T4 · Internet **On** (pretrained SegFormer weights) · Save & Run All (Commit). About 3.2× more training chips than Sen1Floods11, so expect roughly 30 to 55 minutes per run (unmeasured), 6 runs ≈ 3 to 5 h. The loop stops by itself before the session limit; attach its output to a new commit and run again. RAM: about 6 GB for the cached chips.

## 0. Bootstrap

In [ ]:
import sys, os, glob, json, time, shutil, warnings, re, importlib, dataclasses, copy, math
warnings.filterwarnings("ignore")
T_START = time.time()
for _m in [m for m in sys.modules if m == "lgmf" or m.startswith("lgmf.")]: del sys.modules[_m]
importlib.invalidate_caches()
REQUIRED = ["__init__.py", "config.py", "dataset.py", "model.py", "baselines.py", "engine.py", "curriculum.py", "metrics.py",
            "losses.py", "ema.py", "cloud.py", "ramdata.py", "audit.py", "runner.py", "selftest_v8.py"]
def _ver(p):
    m = re.search(r'__version__\s*=\s*"([\d.]+)"', open(p).read())
    return tuple(int(x) for x in m.group(1).split(".")) if m else (0,)
_cands = []
print("every lgmf/ folder found under /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)):
    d = os.path.dirname(p); miss = [f for f in REQUIRED if not os.path.exists(os.path.join(d, f))]
    print(f"   v{'.'.join(map(str, _ver(p)))}  {'COMPLETE' if not miss else 'missing ' + ','.join(miss)}  {d}")
    if not miss: _cands.append((_ver(p), d))
assert _cands, "attach the lgmf_v8 dataset"
_best = max(_cands); assert _best[0] >= (8,), f"best complete lgmf is v{_best[0]}; attach lgmf_v8"
PKG_DIR = _best[1]; sys.path = [os.path.dirname(PKG_DIR)] + [q for q in sys.path if q != os.path.dirname(PKG_DIR)]
try: import segmentation_models_pytorch  # noqa
except ImportError: os.system("pip install -q segmentation-models-pytorch")
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset
import lgmf
assert os.path.dirname(lgmf.__file__) == PKG_DIR, lgmf.__file__
from lgmf.config import LGMFConfig
from lgmf.curriculum import regime_probs
from lgmf.runner import run_experiment, is_done, run_dir
import lgmf.runner as _rn
print("lgmf", lgmf.__version__, "|", PKG_DIR)

# run_experiment also summarises a 'bolivia' split. SenForFlood has no such held-out region, so we give the runner NaN
# placeholders for it (its log line reads res['bolivia']); nothing below ever uses them.
_orig_summarise = _rn.summarise
def _summarise(rows, split):
    out = _orig_summarise(rows, split)
    return out or {r: dict(iou=float("nan"), f1=float("nan"), precision=float("nan"), recall=float("nan"), flood_iou=float("nan")) for r in ("both", "s1_only", "s2_only")}
_rn.summarise = _summarise

PROC = None
for p in sorted(glob.glob("/kaggle/input/**/proc_senfor", recursive=True)):
    if os.path.exists(os.path.join(p, "index.csv")): PROC = p; break
assert PROC, "NB-08 output not found: Add Input -> Notebook Output -> NB-08 (it contains proc_senfor/)"
for f in ("index.csv", "split.json", "norm.json", "card.json", "chips"): assert os.path.exists(os.path.join(PROC, f)), f"{f} missing in {PROC}"
print("SenForFlood data:", PROC)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(CPU: only for the smoke test)")

# ------------------------------ the knobs ------------------------------
EPOCHS = int(os.environ.get("NB15_EPOCHS", 45))        # Study 1 protocol; results live in runs_e<EPOCHS>
SEEDS = [42, 1, 2]
GROUPS_TO_RUN = ["A_recipe"]                           # add "B_factorial" for the optional dropout-vs-consistency runs
TIME_BUDGET_H = 10.5
OUT = "/kaggle/working"
MY_IDS = {"sf_seg_off", "sf_seg_on", "sf_seg_drop_only", "sf_seg_cons_only"}

## 1. Preflight (about 1 min)

In [ ]:
from lgmf import selftest_v8
selftest_v8.run(device=device)

## 2. RESTORE — carry finished NB-15 runs forward (only this notebook's own run ids)

In [ ]:
def restore_previous():
    n = 0
    for p in glob.glob(f"/kaggle/input/**/runs_e{EPOCHS}/*/seed*/result.json", recursive=True):
        rid = p.split(os.sep)[-3]
        if rid not in MY_IDS: continue
        src = os.path.dirname(p); rel = src.split(f"runs_e{EPOCHS}" + os.sep, 1)[1]
        dst = os.path.join(OUT, f"runs_e{EPOCHS}", rel)
        if not os.path.exists(os.path.join(dst, "result.json")):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if os.path.exists(dst): shutil.rmtree(dst)
            shutil.copytree(src, dst); n += 1
    return n
print("restored", restore_previous(), "finished NB-15 run(s) from attached inputs")
for p in sorted(glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/result.json")): print("  have", p.split(f"runs_e{EPOCHS}/")[1].rsplit("/result.json", 1)[0])

## 3. Data: load NB-08's chips into RAM and audit them
The class below presents a SenForFlood split with the same interface as lgmf's `RamDataset`, so `run_experiment`, `chip_counts` and the training loop are the unchanged Study 1 code. Raw float16 chips are stored and standardised on access with the **train-split mean / std from NB-08** (`norm.json`), exactly as the Sen1Floods11 loader does (non-finite → 0 before standardising, then clamp ±10; label −1 → invalid). Augmentation is the same two random flips in the same draw order. There is no crop and no rotation. If chips are smaller than a multiple of 32 they are padded with the channel mean and label −1 (excluded from every metric).

In [ ]:
DATA = PROC
index = pd.read_csv(f"{DATA}/index.csv")
SPLIT = json.load(open(f"{DATA}/split.json"))["chips"]
NORM = json.load(open(f"{DATA}/norm.json")); CARD = json.load(open(f"{DATA}/card.json"))
CH = NORM["channels"]; assert len(CH) == 8 and CH[:2] == ["VV", "VH"], CH
assert CARD["split_by"].startswith("event") and not CARD.get("smoke_test", False), CARD
M = torch.tensor(NORM["mean"], dtype=torch.float32).view(-1, 1, 1); S = torch.tensor(NORM["std"], dtype=torch.float32).view(-1, 1, 1)
print("channels:", CH); print("train-split mean:", np.round(NORM["mean"], 2).tolist()); print("train-split std :", np.round(NORM["std"], 2).tolist())
event_of = lambda cid: cid.split("_")[0]
EV = {k: sorted({event_of(c) for c in v}) for k, v in SPLIT.items()}
print({k: f"{len(SPLIT[k])} chips, {len(EV[k])} events" for k in SPLIT})
for a in ("train", "val", "test"):
    for b in ("train", "val", "test"):
        if a < b: assert not (set(EV[a]) & set(EV[b])), f"events shared between {a} and {b}: {set(EV[a]) & set(EV[b])}"
print("events are disjoint across train / val / test")
print("  val events :", EV["val"]); print("  test events:", EV["test"])
print("  sample chip ids:", SPLIT["train"][:3])

# chip sizes (header only)
shapes = {}
for k in SPLIT:
    for c in SPLIT[k]:
        sh = tuple(np.load(f"{DATA}/chips/{c}_y.npy", mmap_mode="r").shape); shapes[sh] = shapes.get(sh, 0) + 1
print("chip shapes:", shapes)
SIZE = int(math.ceil(max(max(s) for s in shapes) / 32) * 32); print("working size:", SIZE)

class SenForRam(Dataset):
    def __init__(self, ids, augment=False):
        self.ids, self.augment = list(ids), augment
        self.x, self.y = [], []
        self.nonfinite = np.zeros(8); self.npix = 0; self.lab = np.zeros(3)   # -1, 0, 1
        for c in self.ids:
            x = np.load(f"{DATA}/chips/{c}_x.npy"); y = np.load(f"{DATA}/chips/{c}_y.npy")
            assert x.ndim == 3 and x.shape[0] == 8 and x.shape[1:] == y.shape, (c, x.shape, y.shape)
            self.nonfinite += (~np.isfinite(x.astype("float32"))).sum((1, 2)); self.npix += y.size
            for i, v in enumerate((-1, 0, 1)): self.lab[i] += (y == v).sum()
            h, w = y.shape
            if (h, w) != (SIZE, SIZE):
                xp = np.broadcast_to(np.array(NORM["mean"], dtype="float16")[:, None, None], (8, SIZE, SIZE)).copy(); xp[:, :h, :w] = x; x = xp
                yp = np.full((SIZE, SIZE), -1, dtype="int8"); yp[:h, :w] = y; y = yp
            self.x.append(np.ascontiguousarray(x.astype("float16"))); self.y.append(np.ascontiguousarray(y.astype("int8")))
    def __len__(self): return len(self.ids)
    def _one(self, i):
        x = torch.from_numpy(self.x[i].astype("float32")); x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        x = torch.clamp((x - M) / (S + 1e-6), -10.0, 10.0)
        y = torch.from_numpy(self.y[i].astype("int64"))
        return x[:2], x[2:], (y > 0).float(), (y >= 0).float()
    def __getitem__(self, i):
        s1, s2, label, valid = self._one(i)
        if self.augment:     # same draw order as RamDataset / Sen1Floods11._augment
            if torch.rand(1).item() < 0.5: s1, s2, label, valid = [t.flip(-1) for t in (s1, s2, label, valid)]
            if torch.rand(1).item() < 0.5: s1, s2, label, valid = [t.flip(-2) for t in (s1, s2, label, valid)]
        return dict(s1=s1, s2=s2, label=label, valid=valid, jrc=torch.zeros_like(label))
    def batch(self, ii, variant="zero"):
        assert variant == "zero"
        P = [self._one(i) for i in ii]; st = lambda k: torch.stack([p[k] for p in P])
        return dict(s1=st(0), s2=st(1), label=st(2), valid=st(3), jrc=torch.zeros_like(st(2)))

t0 = time.time()
TRAIN = SenForRam(SPLIT["train"], augment=True)
EVAL = {k: SenForRam(SPLIT[k]) for k in ("val", "test")}
print({"train": len(TRAIN), "val": len(EVAL["val"]), "test": len(EVAL["test"])}, f"cached in {time.time()-t0:.0f}s")

In [ ]:
# audit of what the models will see
print("non-finite raw pixels per channel (train chips, %):", {k: round(float(v), 4) for k, v in zip(CH, 100 * TRAIN.nonfinite / max(TRAIN.npix, 1))})
print("label share in train chips: no-data %.3f  land %.3f  water %.3f" % tuple(TRAIN.lab / TRAIN.lab.sum()))
FLOOR = {}
for k, ds in (("train", TRAIN), ("val", EVAL["val"]), ("test", EVAL["test"])):
    w = sum(float((y == 1).sum()) for y in ds.y); v = sum(float((y >= 0).sum()) for y in ds.y); FLOOR[k] = w / v
    print(f"{k:5s}: water fraction of valid pixels {FLOOR[k]:.3f}  -> always-water IoU = {FLOOR[k]:.3f}, always-land IoU = 0.000")
print("water fraction per chip (test): min %.3f, median %.3f, max %.3f; chips without any water: %d of %d" % (
    min(float((y == 1).sum() / max((y >= 0).sum(), 1)) for y in EVAL['test'].y), np.median([float((y == 1).sum() / max((y >= 0).sum(), 1)) for y in EVAL['test'].y]),
    max(float((y == 1).sum() / max((y >= 0).sum(), 1)) for y in EVAL['test'].y), sum(int((y == 1).sum() == 0) for y in EVAL['test'].y), len(EVAL['test'])))
print("\nchips per event:"); print(pd.DataFrame({k: pd.Series([event_of(c) for c in v]).value_counts() for k, v in SPLIT.items()}).fillna(0).astype(int).sort_index().to_string())

## 4. The run table

In [ ]:
def make_cfg(overrides, seed):
    c = LGMFConfig()
    c.epochs, c.lr, c.batch_size, c.ema_decay = EPOCHS, 3e-4, 8, 0.99
    c.loss_type, c.consistency_weight, c.num_workers = "focal_tversky", 0.5, 0
    c.s1_mean, c.s1_std, c.s2_mean, c.s2_std = NORM["mean"][:2], NORM["std"][:2], NORM["mean"][2:], NORM["std"][2:]
    c.seed = seed
    for k, v in overrides.items(): setattr(c, k, v)
    return c
NODROP = dict(use_curriculum=False, fixed_regime_probs=(1.0, 0.0, 0.0))
SEG = dict(kind="concat", arch="segformer", encoder="mit_b0")
RUNS = [   # (run_id, group, model spec, config overrides, same recipe as on Sen1Floods11)    selection: mean val IoU over the 3 regimes
    ("sf_seg_off",       "A_recipe",    SEG, dict(NODROP, consistency_weight=0.0), "= seg_plain"),
    ("sf_seg_on",        "A_recipe",    SEG, {},                                   "= seg_full_ref (the paper's recipe)"),
    ("sf_seg_drop_only", "B_factorial", SEG, dict(consistency_weight=0.0),         "curriculum, no consistency (= segformer_noconsist)"),
    ("sf_seg_cons_only", "B_factorial", SEG, dict(NODROP),                         "= seg_consonly"),
]
assert {r[0] for r in RUNS} == MY_IDS
_c = make_cfg({}, 42)
print("curriculum cumulative exposure (both, SAR-only, optical-only):", np.round(np.mean([regime_probs(_c, e) for e in range(EPOCHS)], 0), 4))
print(pd.DataFrame([dict(run=r, group=g, consistency=o.get("consistency_weight", 0.5), dropout="no" if o.get("use_curriculum") is False else "curriculum", same_as=n) for r, g, s, o, n in RUNS]).to_string(index=False))

## 5. Train (seed-major; resumable; stops before the session limit)

In [ ]:
todo = [(rid, g, spec, ov, sd) for sd in SEEDS for rid, g, spec, ov, _ in RUNS if g in GROUPS_TO_RUN and not is_done(OUT, EPOCHS, rid, sd)]
print(f"{len(todo)} run(s) to do in {GROUPS_TO_RUN}")
dur = []
for rid, g, spec, ov, sd in todo:
    est = (np.mean(dur) if dur else 60.0) * 1.15
    left = TIME_BUDGET_H * 60 - (time.time() - T_START) / 60
    if est > left:
        print(f"STOP: next run ({rid}, seed {sd}) needs ~{est:.0f} min, only {left:.0f} min of budget left. "
              f"Save this version, attach its output to a new commit, and run again (finished runs are skipped).")
        break
    t1 = time.time()
    run_experiment(rid, make_cfg(ov, sd), spec, TRAIN, EVAL["val"], {"val": EVAL["val"], "test": EVAL["test"]}, OUT, device, select="mean")
    dur.append((time.time() - t1) / 60)
print("minutes per run:", round(float(np.mean(dur)), 1) if dur else None)
print(f"done; {len(dur)} run(s) this session, {(time.time()-T_START)/3600:.1f} h elapsed")

## 6. Results — val and test, mean ± std over seeds (ddof = 1), with the worst seed, the predicted-to-true water ratio and the trivial floor
`ratio` = predicted water pixels / true water pixels (near 0 = silent, near 1 = right amount, far above 1 = over-predicting). 

In [ ]:
RJ = [json.load(open(p)) for p in sorted(glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/result.json"))]
assert RJ, "no finished runs"
PC = pd.concat([pd.read_csv(p).assign(run=lambda d: d.config) for p in sorted(glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/perchip.csv.gz"))], ignore_index=True)
PC = PC[PC.split.isin(["val", "test"]) & (PC.variant == "zero")]
PC[["tp", "fp", "fn", "tn"]] = PC[["tp", "fp", "fn", "tn"]].astype(float)
def pooled(df, keys):
    g = df.groupby(keys)[["tp", "fp", "fn", "tn"]].sum().reset_index()
    g["iou"] = g.tp / (g.tp + g.fp + g.fn + 1e-6); g["ratio"] = (g.tp + g.fp) / (g.tp + g.fn + 1e-6); return g
PS = pooled(PC, ["run", "seed", "split", "regime"]); PS.to_csv(f"{OUT}/nb15_per_seed.csv", index=False)
# the pooled-from-chips IoU must equal the runner's own summary
chk = max(abs(PS[(PS.run == r["run_id"]) & (PS.seed == r["seed"]) & (PS.split == "test") & (PS.regime == "s1_only")].iou.iloc[0] - r["test"]["s1_only"]["iou"]) for r in RJ)
print(f"check: per-chip pooled IoU vs runner summary, max |difference| = {chk:.1e}"); assert chk < 1e-4
ORDER = [r for r in ["sf_seg_off", "sf_seg_cons_only", "sf_seg_drop_only", "sf_seg_on"] if r in set(PS.run)]
pd.set_option("display.width", 250)
def table(split):
    g = PS[PS.split == split].groupby(["run", "regime"]).agg(m=("iou", "mean"), s=("iou", lambda x: x.std(ddof=1) if len(x) > 1 else 0.0), lo=("iou", "min"), rt=("ratio", "mean"), n=("iou", "count")).reset_index()
    out = {}
    for reg in ("both", "s1_only", "s2_only"):
        d = g[g.regime == reg].set_index("run")
        out[reg] = d.apply(lambda r: f"{r.m:.3f}±{r.s:.3f} (min {r.lo:.3f}, ratio {r.rt:.2f})", axis=1)
    t = pd.DataFrame(out).reindex(ORDER); t["n"] = g.groupby("run").n.max(); return t
for sp in ("val", "test"):
    print(f"\n=== {sp.upper()}  IoU mean±std (min over seeds, predicted/true water ratio).   trivial floor: always-water {FLOOR[sp]:.3f}, always-land 0.000 ===")
    t = table(sp); print(t.to_string()); t.to_csv(f"{OUT}/nb15_table_{sp}.csv")
print("\nper-seed optical-only and SAR-only IoU (test):")
print(PS[(PS.split == "test") & PS.regime.isin(["s1_only", "s2_only"])].pivot_table(index=["run", "regime"], columns="seed", values="iou").reindex(ORDER, level=0).round(3).to_string())

## 7. Paired contrasts (same seed) and the pre-set classification
`same` = identical sign in all seeds. The classification uses the **test** split and the rule above; the primary regime is optical-only (`s2_only`).

In [ ]:
def ps(run, split, reg): return PS[(PS.run == run) & (PS.split == split) & (PS.regime == reg)].set_index("seed").iou
def pair(A, B, split, reg):
    a, b = ps(A, split, reg), ps(B, split, reg); k = sorted(set(a.index) & set(b.index))
    return (a[k] - b[k]) if k else None
def classify(d, thr=0.03):
    if d is None: return "n/a"
    m = d.mean(); same = bool((d > 0).all() or (d < 0).all())
    return ("helps" if m > 0 else "hurts") if (abs(m) > thr and same) else "no detectable effect"
CONTRASTS = [("recipe on - off (the paper's question)", "sf_seg_on", "sf_seg_off"),
             ("dropout alone - off", "sf_seg_drop_only", "sf_seg_off"), ("consistency alone - off", "sf_seg_cons_only", "sf_seg_off"),
             ("recipe on - dropout alone (what consistency adds)", "sf_seg_on", "sf_seg_drop_only"),
             ("recipe on - consistency alone (what the curriculum adds)", "sf_seg_on", "sf_seg_cons_only")]
REGN = {"s2_only": "optical-only (PRIMARY)", "s1_only": "SAR-only", "both": "both"}
CT = []
for title, A, B in CONTRASTS:
    if A not in ORDER or B not in ORDER: continue
    print(f"\n{title}   [{A} - {B}]")
    for reg in ("s2_only", "s1_only", "both"):
        line = f"   {REGN[reg]:22s}"
        for sp in ("val", "test"):
            d = pair(A, B, sp, reg)
            line += f"  {sp}: {d.mean():+.3f} [" + " ".join(f"{s}:{v:+.3f}" for s, v in d.items()) + "]" + (" same" if (d > 0).all() or (d < 0).all() else "") + "   "
        print(line)
        d = pair(A, B, "test", reg); CT.append(dict(contrast=title, regime=REGN[reg], mean_diff=round(float(d.mean()), 3), min_diff=round(float(d.min()), 3), max_diff=round(float(d.max()), 3), verdict=classify(d)))
CT = pd.DataFrame(CT); print("\n=== Rule 2 (test): helps / hurts only if |mean diff| > 0.03 AND same sign in all seeds ==="); print(CT.to_string(index=False)); CT.to_csv(f"{OUT}/nb15_contrasts.csv", index=False)
if "sf_seg_on" in ORDER and "sf_seg_off" in ORDER:
    v = CT[CT.contrast.str.startswith("recipe on - off")].set_index("regime").verdict
    prim, others = v["optical-only (PRIMARY)"], [v["SAR-only"], v["both"]]
    print("\nPRE-SET READING of rule 3:",
          "the recipe TRANSFERS (optical-only helps, nothing hurts)." if prim == "helps" and "hurts" not in others else
          "NOT shown to transfer: " + f"optical-only = {prim}; SAR-only = {v['SAR-only']}; both = {v['both']}.")

## 8. Per event (test and val): is the effect general or carried by one flood?
Counts are pooled over the 3 seeds within each event, then IoU is computed (never a mean of per-chip IoUs).

In [ ]:
PE = pooled(PC, ["run", "split", "event", "regime"])
for sp in ("test", "val"):
    for reg in ("s2_only", "s1_only"):
        t = PE[(PE.split == sp) & (PE.regime == reg)].pivot(index="event", columns="run", values="iou").reindex(columns=ORDER).round(3)
        t.insert(0, "chips", PC[(PC.split == sp) & (PC.regime == "both") & (PC.run == ORDER[0]) & (PC.seed == PC.seed.iloc[0])].groupby("event").size())
        print(f"\n{REGN[reg]} IoU per {sp} event (seeds pooled):"); print(t.to_string())
        if {"sf_seg_on", "sf_seg_off"} <= set(t.columns):
            gap = (t.sf_seg_on - t.sf_seg_off).dropna()
            print(f"   on - off per event: median {gap.median():+.3f}, range {gap.min():+.3f} to {gap.max():+.3f}, positive in {int((gap > 0).sum())}/{len(gap)} events")
PE.to_csv(f"{OUT}/nb15_per_event.csv", index=False)

## 9. For orientation only: NB-09 (different protocol) and Sen1Floods11 (same protocol)
**Do not put the NB-09 column in the paper.** It is printed to see whether the conclusion survives the change of protocol, which is a robustness statement in itself. The Sen1Floods11 columns are the NB-12 runs `seg_plain` and `seg_full_ref` (same code and protocol), if the NB-12 output is attached.

In [ ]:
NB09 = {"off": {"both": 0.650, "optical": 0.041, "sar": 0.615}, "on": {"both": 0.650, "optical": 0.190, "sar": 0.644}}   # NB-09 v7, 90 epochs, CE + symmetric KL, lr 6e-5, 256 crops, 3 seeds, test
rows = []
for reg, key in (("both", "both"), ("s2_only", "optical"), ("s1_only", "sar")):
    r = dict(regime=REGN[reg])
    for tag, run in (("off", "sf_seg_off"), ("on", "sf_seg_on")):
        r[f"NB-15 {tag}"] = round(float(ps(run, "test", reg).mean()), 3) if run in ORDER else None
    r["NB-15 on-off"] = round(r["NB-15 on"] - r["NB-15 off"], 3) if r["NB-15 on"] is not None and r["NB-15 off"] is not None else None
    r["NB-09 off"], r["NB-09 on"] = NB09["off"][key], NB09["on"][key]; r["NB-09 on-off"] = round(NB09["on"][key] - NB09["off"][key], 3)
    S11 = {}
    for tag, rid in (("off", "seg_plain"), ("on", "seg_full_ref")):
        v = [json.load(open(p))["test"][reg]["iou"] for p in glob.glob(f"/kaggle/input/**/runs_e{EPOCHS}/{rid}/seed*/result.json", recursive=True)]
        S11[tag] = float(np.mean(v)) if v else None
    r["Sen1Floods11 off"], r["Sen1Floods11 on"] = (round(S11["off"], 3) if S11["off"] is not None else None), (round(S11["on"], 3) if S11["on"] is not None else None)
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

## 10. Measured regime exposure and run times

In [ ]:
rows = []
for r in RJ:
    rows.append(dict(run=r["run_id"], seed=r["seed"], measured_both=r["measured_cumulative_shares"][0], measured_sar=r["measured_cumulative_shares"][1], measured_opt=r["measured_cumulative_shares"][2],
                     scheduled_both=r["scheduled_cumulative_probs"][0], scheduled_sar=r["scheduled_cumulative_probs"][1], scheduled_opt=r["scheduled_cumulative_probs"][2],
                     best_epoch=r["best_epoch"], train_min=r["train_minutes"]))
X = pd.DataFrame(rows); g = X.groupby("run").mean(numeric_only=True).drop(columns="seed"); print(g.round(4).to_string()); g.to_csv(f"{OUT}/nb15_exposure.csv")
print("\nbest epoch per run (a best epoch near 45 = still improving; near the start = the selection signal is noisy with only 3 validation events):")
print(X.pivot(index="run", columns="seed", values="best_epoch").to_string())

## 11. What to send back
When the commit has finished: send me the printed output of sections 3 and 5–10 (or the executed `.ipynb`). If the loop printed `STOP: …`, save the version, attach its output to a new commit of this notebook and run again.
Optional: change `GROUPS_TO_RUN` to `["A_recipe", "B_factorial"]` (6 more runs) to see whether dropout and consistency are substitutes on this dataset too.